In [6]:
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
import os
import io
import builtins
import time
import re
from itertools import product
import dash
from dash import dcc
from dash import html 
from dash import dash_table
import dash_bootstrap_components as dbc


from dash.dependencies import Input, Output
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 63)

In [7]:
python_exports = "C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports"
current_save = "Python Player Prediction Data"
all_players_file = "all_players_data.csv"

In [8]:
folder_path = python_exports + "/" + current_save
all_players_file_path = folder_path + "/" + all_players_file
all_folder_files = os.listdir(folder_path)


In [9]:
all_html_files = [f for f in all_folder_files if f.endswith('.html')]    
all_html_files = [w.replace('.html', '') + ' 00:00:01' for w in all_html_files]


In [10]:
if not os.path.exists(all_players_file_path):
    exported_dates = pd.DataFrame({'Date': [1, 2]})

else:
    # If the file exists, read the existing data and make a copy    
    exported_dates = pd.read_csv(all_players_file_path)

exported_dates = exported_dates.Date.unique().tolist()


In [11]:
for file in all_html_files:
    if file not in exported_dates:
        file_name = file.replace(' 00:00:01','')
        full_path = folder_path + '/' + file_name + '.html'
        fm_export= bs(open(full_path,encoding = "utf8"), "html.parser").find('table')
        raw_list = pd.read_html(io.StringIO(str(fm_export)))
        raw_table = pd.DataFrame(raw_list[0])
        date = file_name.split('.')[0] +  ' 00:00:01'
        raw_table['Date'] = pd.to_datetime(date, format = "%Y-%m-%d %H:%M:%S")
        raw_table['DoB'] = pd.to_datetime(raw_table["DoB"].str.split(" ").str[0], format="%d/%m/%Y")
        raw_table['DoB'] = raw_table['DoB'] + pd.Timedelta(seconds=1)
        raw_table['Age'] = (raw_table['Date']-raw_table['DoB']).dt.days/365.25
        raw_table['Acc'] = pd.to_numeric(raw_table['Acc'], errors='coerce')

        raw_table = raw_table.dropna(subset=['Acc'])
        if not os.path.exists(all_players_file_path):
            raw_table.to_csv(all_players_file_path, index=False)
            print(f"Created file: {all_players_file_path}")
        else:
            # If the file exists, read the existing data and make a copy
            timestr = time.strftime("%Y%m%d-%H%M%S")
            existing_df = pd.read_csv(all_players_file_path)
            csv_filename = f'C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/{current_save}/all_players_data_{timestr}.csv'
            existing_df.to_csv(csv_filename, index=False)

            # Union the existing DataFrame with the new DataFrame
            combined_df = pd.concat([existing_df, raw_table]).drop_duplicates().reset_index(drop=True)

            # Save the combined DataFrame back to the CSV file
            combined_df.to_csv(all_players_file_path, index=False)
            print(f"Updated file: {all_players_file_path}")


ValueError: time data "01-01-24 00:00:01" doesn't match format "%Y-%m-%d %H:%M:%S", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
full_path = folder_path + '/' + 'all_players_data' + '.csv'

In [ ]:
raw_table = pd.read_csv(full_path)


In [ ]:
def process_segment(segment):
    segment = segment.replace("D/","Z/").replace("D ",("Z "))
    sides = re.findall(r'\((.*?)\)', segment)
    print(sides)
    sides = sides[0]    
    print(sides)
    lines = segment.replace("("+sides+")","").replace("/","").replace("AM","A").replace("DM","O").replace('ST','F').replace("WB","O").replace("Z","D").strip()
    print(lines)
    result = [f"{x}{y}" for x, y in product(lines, sides)] 
    print(result)
        
        
    return result

In [ ]:
def combine_segment(segments):
    result = []
    segments = segments.replace('GK','G (C)')
    segments = re.sub(r'\bDM\b(,?)', r'DM (C)\1', segments)
    for segment in segments.split(','):
        result.extend(process_segment(segment.strip()))
    return result

In [ ]:
def apply_combinations(df, column):

    df["Positions"] = df[column].apply(combine_segment)
    df['Positions'] = df['Positions'].apply(replace_values)

    return df


In [ ]:
def replace_values(lst):
    new_lst = []
    for item in lst:
        item = item.replace('O', 'DM')
        item = item.replace('GC', 'GK')
        item = item.replace('FC', 'ST')
        item = item.replace('A', 'AM')
        item = item.replace('DMR', 'WBR')
        item = item.replace('DML', 'WBL')
        item = item.replace('DMC', 'DM')


        new_lst.append(item)
    return new_lst

In [ ]:
df = apply_combinations(raw_table, 'Position')


In [ ]:
unique_values = set([item for sublist in df['Positions'] for item in sublist])


In [ ]:
for position in unique_values:
    df[position] = df.apply(lambda x: 1 if position in x["Positions"] else 0, axis=1)


In [ ]:
latest_export_date = df['Date'].max()
df['gk_rating'] = ((5 * (df['Agi']+df['Ref']+ df['Cmd']+df['Cnt'])) + (3 * (df['1v1']+df['Ant']+df['Kic']+df['Pos'])) +(1 * (df['Acc']+df['Aer']+df['Dec']+df['Fir']+df['Han']+df['Pas']+df['TRO']+df['Vis'])))/40
df['of_rating'] = (((df['Acc']+ df['Pac'])*5) +  ((df['Dri']+ df['Ant'] + df['Cnt']+ df['Agi']+df['Bal']+ df['Jum'])*3) + (df['Det']+ df['Wor'] + df['Sta']+ df['Nat.1']))/32
df['rating'] = df['gk_rating'].where(df['Positions'] == 'GK', df['of_rating'])
df['physical'] = df['Acc']+df['Pac'] + df['Jum']
earliest_ratings = df.loc[df.groupby('UID')['Date'].idxmin(), ['UID', 'rating','physical','CA','PA']].rename(columns={'rating': 'first_rating','physical': 'first_physical','CA': 'first_CA','PA': 'first_PA'})
df = df.merge(earliest_ratings, on='UID', how='left')

df['rating_growth'] = df['rating'] - df['first_rating']
df['physical_growth'] = df['physical'] - df['first_physical']
df['CA_growth'] = df['CA'] - df['first_CA']

df['age_at_export'] = df['Age'].apply(np.floor)
df['current_actual_age'] = df.groupby(['UID'])['age_at_export'].transform('last')
df['is_latest_export'] = df['Date'] == latest_export_date
df_max_date = df[df['Date'] == latest_export_date]
df_max_date['in_latest_export'] = True
df = pd.merge(df, df_max_date, how="left", on=["UID"],suffixes = [None,'_current'])
df=df.round(2)




In [17]:
df['Date'] = pd.to_datetime(df['Date'])

july_first_dates = df[df['Date'].dt.month == 7][df['Date'].dt.day == 1]
latest_july_first = july_first_dates['Date'].max()
start_of_latest_season_df = df[(df['Date'] == latest_july_first) ]
start_of_latest_season_df = start_of_latest_season_df[['UID','age_at_export']]
df = df.merge(start_of_latest_season_df, on='UID', how='left',suffixes=('','_start_of_season'))


C:\Users\jackj\AppData\Local\Temp\ipykernel_17816\4201536246.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  july_first_dates = df[df['Date'].dt.month == 7][df['Date'].dt.day == 1]


In [18]:
columns_to_check = ['GK', 'DC', 'WBR', 'WBL', 'DM', 'AMC', 'ST']
df['Positions'] = df.apply(lambda row: [col for col in columns_to_check if row[col] == 1], axis=1)
df['Positions'] = df['Positions'].apply(lambda x: ', '.join(x))
df['Positions_filter'] = df['Positions']+', All'
df['All'] = 1
team_df = df[df['Club'].str.contains(current_save, case=False, na=False)]
team_df = team_df[team_df['is_latest_export']==True]
df = df[['Name','Club','current_actual_age','Positions','Transfer Value','Positions_filter','GK','DC','WBR','DM','WBL','AMC','ST','All','rating','rating_growth','physical','physical_growth','Pac','Acc','Jum','age_at_export','is_latest_export','in_latest_export','age_at_export_start_of_season']]

In [19]:


app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Callback function to update the datatable
@app.callback(
    Output(component_id='my_squad', component_property='data'),
    [
        Input(component_id='age_range_slider', component_property='value'),
        Input(component_id='position_filter', component_property='value'),
        Input(component_id='is_latest_export_filter', component_property='value'),
        Input(component_id='club_filter', component_property='value')
    ]
)
def update_datatable(selected_age_range, selected_positions, is_latest_export, club_filter):
    # Make a copy of the global df to avoid modifying it directly
    df_filtering = df.copy()
    df_filtering['Positions_filter'] = df_filtering['Positions_filter'].str.split(', ')

    # Filter the DataFrame based on the selected positions
    if 'All' in selected_positions:
        filtered_df = df_filtering
    else:
        filtered_df = df_filtering[df_filtering['Positions_filter'].apply(
            lambda x: any(pos in x for pos in selected_positions))]

    # Convert list in 'Positions' column to comma-separated strings for display
    filtered_df['Positions_filter'] = filtered_df['Positions_filter'].apply(lambda x: ', '.join(x))

    # Filter by 'is_latest_export'
    filtered_df = filtered_df[filtered_df['is_latest_export'].isin(is_latest_export)]

    # Filter by age range
    filtered_df = filtered_df[
        (selected_age_range[0] <= filtered_df['age_at_export_start_of_season']) &
        (filtered_df['age_at_export_start_of_season'] <= selected_age_range[1])
    ]

    # Filter by club name
    if club_filter:  # Only filter if club_filter is not empty
        filtered_df = filtered_df[filtered_df['Club'].str.contains(club_filter, case=False, na=False)]


    # Return the filtered DataFrame as a list of dictionaries
    return filtered_df.to_dict('records')


# Define the layout of the app
app.layout = html.Div([
    dbc.Row(html.H1("Web App", style={'text-align': 'center'})),

    dbc.Row([
        dbc.Col(html.Div([
            dcc.RangeSlider(15, 40,
                            value=[15, 40],
                            id='age_range_slider'),
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Dropdown(id="position_filter",
                         options=[{'label': pos, 'value': pos} for pos in ['GK', 'DC', 'WBR', 'WBL', 'DM', 'AMC', 'ST', 'All']],
                         multi=True,
                         value=['All']),  # Default value
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Dropdown(id="is_latest_export_filter",
                         options=[{'label': 'True', 'value': True}, {'label': 'False', 'value': False}],
                         multi=True,
                         value=[True]),  # Default value
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Input(id="club_filter",
                      type='text',
                      placeholder='Enter a club name...',
                      debounce=True,
                      value=""),  # Default value
        ]), width=3),
    ], className='row'),

    html.Br(),
    dbc.Container(
        dbc.Row(
            dbc.Col(
                html.Div(
                    id='output_container',
                    children=[
                        dash_table.DataTable(
                            columns=[{"name": i, "id": i} for i in df.columns],  # Include Positions column
                            data=df.to_dict('records'),
                            page_size=15,
                            id='my_squad',
                            filter_action="native",
                            sort_action="native",
                            sort_mode="multi",
                            row_selectable="multi",
                            hidden_columns=['is_latest_export', 'Positions_filter','GK','DC','WBR','WBL','DM','AMC','All','ST']  # Adjust hidden columns
                        )
                    ]
                ),
            ),
            className='row'
        ), fluid=True
    )
])




In [20]:
if __name__ == '__main__':
    app.run(debug=True,jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [21]:
team_df = team_df[['UID','Name','GK','DC','WBR','DM','WBL','AMC','ST','All','rating','age_at_export_start_of_season']]

In [22]:
import pandas as pd
from itertools import permutations

# Define the global position order for sorting
position_order = ["GK", "DC", "DC", "DC", "WBR", "DM", "DM", "WBL", "AMC", "AMC", "ST"]  # Fixed order

def assign_large_teams(team_df):
    position_groups = ["DC", "WBR", "DM", "WBL", "AMC", "ST"]  # Changed AM to AMC
    permutations_list = list(permutations(position_groups))
    
    best_teams = None
    best_rating_sum = 0

    # Iterate through each permutation of position groups
    for perm in permutations_list:
        team_df_copy = team_df.copy()
        team_df_copy['Assigned'] = False

        # Large teams with all required positions
        team_1 = []
        team_2 = []

        team_1_rating = 0
        team_2_rating = 0

        # Define position slots for the two large teams
        team_slots = {
            "GK": 2, "DC": 6, "WBR": 2, "WBL": 2, "DM": 4, "AMC": 4, "ST": 2
        }

        # Assign players for the given permutation
        for position in perm:
            required_slots = team_slots[position]
            
            # Get the top players for this position who are not assigned
            candidates = team_df_copy[(team_df_copy[position]) & (~team_df_copy['Assigned'])]
            candidates = candidates.sort_values(by='rating', ascending=False)

            assigned_count = 0
            for _, player in candidates.iterrows():
                if assigned_count < required_slots // 2:
                    team_1.append((player.UID, player.Name, player.age_at_export_start_of_season, player.rating, position))
                    team_1_rating += player.rating
                elif assigned_count < required_slots:
                    team_2.append((player.UID, player.Name, player.age_at_export_start_of_season, player.rating, position))
                    team_2_rating += player.rating

                # Mark the player as assigned
                team_df_copy.loc[team_df_copy.UID == player.UID, 'Assigned'] = True
                assigned_count += 1

                # Break when all slots for this position are filled
                if assigned_count == required_slots:
                    break

        # Assign Goalkeepers explicitly (only one per team)
        gk_candidates = team_df_copy[team_df_copy['GK'] & (~team_df_copy['Assigned'])]
        gk_candidates = gk_candidates.sort_values(by='rating', ascending=False)

        assigned_gks = 0
        for _, player in gk_candidates.iterrows():
            if assigned_gks < 1:  # Assign one GK to Team 1
                team_1.append((player.UID, player.Name, player.age_at_export_start_of_season, player.rating, "GK"))
                team_1_rating += player.rating
            elif assigned_gks < 2:  # Assign one GK to Team 2
                team_2.append((player.UID, player.Name, player.age_at_export_start_of_season, player.rating, "GK"))
                team_2_rating += player.rating

            # Mark the player as assigned
            team_df_copy.loc[team_df_copy.UID == player.UID, 'Assigned'] = True
            assigned_gks += 1

            if assigned_gks == 2:
                break

        # Calculate the total rating for both teams
        total_rating = team_1_rating + team_2_rating
        
        # Check if this permutation results in a better rating sum
        if total_rating > best_rating_sum:
            best_rating_sum = total_rating
            best_teams = {
                'Permutation': perm,
                'Team_1': team_1,
                'Team_2': team_2,
                'Team_1_Rating': team_1_rating,
                'Team_2_Rating': team_2_rating
            }

    # Return the best teams (Team 1 and Team 2)
    return best_teams


def assign_age_group_teams(team_df, best_teams):
    # Exclude players already assigned to Team 1 and Team 2
    assigned_players = set([player[0] for player in best_teams['Team_1']] + [player[0] for player in best_teams['Team_2']])
    remaining_players_df = team_df[~team_df['UID'].isin(assigned_players)].copy()

    # Define the age groups (16 to 21)
    age_groups = range(16, 22)
    
    # Define the position groups (same as before)
    position_groups = ["DC", "WBR", "DM", "WBL", "AMC", "ST"]  # Changed AM to AMC
    team_slots = {
        "GK": 1, "DC": 3, "WBR": 1, "WBL": 1, "DM": 2, "AMC": 2, "ST": 1
    }

    best_teams_by_age = {}

    # Iterate over each age group (16 to 21)
    for age in age_groups:
        age_group_df = remaining_players_df[remaining_players_df['age_at_export_start_of_season'] == age].copy()
        
        if age_group_df.empty:
            continue  # Skip if there are no players in this age group
        
        best_team_for_age = None
        best_rating_sum = 0
        
        # Generate all possible permutations of the position groups
        permutations_list = list(permutations(position_groups))
        
        for perm in permutations_list:
            team_df_copy = age_group_df.copy()
            team_df_copy['Assigned'] = False

            # Create team
            team = []
            team_rating = 0

            # Assign players for the given permutation
            for position in perm:
                required_slots = team_slots[position]
                
                # Get the top players for this position who are not assigned
                candidates = team_df_copy[(team_df_copy[position]) & (~team_df_copy['Assigned'])]
                candidates = candidates.sort_values(by='rating', ascending=False)

                assigned_count = 0
                for _, player in candidates.iterrows():
                    if assigned_count < required_slots:
                        team.append((player.UID, player.Name, player.age_at_export_start_of_season, player.rating, position))
                        team_rating += player.rating
                        team_df_copy.loc[team_df_copy.UID == player.UID, 'Assigned'] = True
                        assigned_count += 1

                # Fill empty slots for this position
                if assigned_count < required_slots:
                    for _ in range(required_slots - assigned_count):
                        team.append(("Empty", "Empty", age, 0, position))

            # Assign the best GK if possible
            gk_candidates = team_df_copy[team_df_copy['GK'] & (~team_df_copy['Assigned'])]
            gk_candidates = gk_candidates.sort_values(by='rating', ascending=False)
            assigned_gk = False

            for _, player in gk_candidates.iterrows():
                team.append((player.UID, player.Name, player.age_at_export_start_of_season, player.rating, "GK"))
                team_rating += player.rating
                team_df_copy.loc[team_df_copy.UID == player.UID, 'Assigned'] = True
                assigned_gk = True
                break

            if not assigned_gk:
                team.append(("Empty", "Empty", age, 0, "GK"))

            # Calculate the total rating for the team
            if team_rating > best_rating_sum:
                best_rating_sum = team_rating
                best_team_for_age = {
                    'Age': age,
                    'Team': team,
                    'Team_Rating': team_rating
                }

        # Store the best team for this age group
        if best_team_for_age:
            best_teams_by_age[age] = best_team_for_age

    return best_teams_by_age


# Main execution
# Assuming `team_df` is already defined
best_teams = assign_large_teams(team_df)

# Assign the remaining players to age group teams
best_age_group_teams = assign_age_group_teams(team_df, best_teams)

# Print results
print("\nBest Team 1:")
team_1_sorted = sorted(best_teams['Team_1'], key=lambda x: position_order.index(x[4]))  # Sort based on position order
print(f"{'Position':<10}{'Name':<20}{'Age':<5}{'Rating':<7}")
for _, name, age, rating, pos in team_1_sorted:
    print(f"{pos:<10}{name:<20}{age:<5}{rating:<7}")

print(f"\nTotal Rating for Team 1: {best_teams['Team_1_Rating']}")

print("\nBest Team 2:")
team_2_sorted = sorted(best_teams['Team_2'], key=lambda x: position_order.index(x[4]))  # Sort based on position order
print(f"{'Position':<10}{'Name':<20}{'Age':<5}{'Rating':<7}")
for _, name, age, rating, pos in team_2_sorted:
    print(f"{pos:<10}{name:<20}{age:<5}{rating:<7}")

print(f"\nTotal Rating for Team 2: {best_teams['Team_2_Rating']}")

# Print the best teams for each age group
for age in sorted(best_age_group_teams.keys(), reverse=True):
    best_team = best_age_group_teams[age]
    print(f"\nBest Team for Age {age}:")
    print(f"Total Team Rating: {best_team['Team_Rating']}")
    
    team_sorted = sorted(best_team['Team'], key=lambda x: position_order.index(x[4]))  # Sort based on position order
    print(f"{'Position':<10}{'Name':<20}{'Age':<5}{'Rating':<7}")
    for _, name, age, rating, pos in team_sorted:
        print(f"{pos:<10}{name:<20}{age:<5}{rating:<7}")



Best Team 1:
Position  Name                Age  Rating 
GK        Mikołaj Biegański   25.0 12.34  
DC        João Fonseca        20.0 14.72  
DC        Giorgio Scalvini    23.0 14.72  
DC        El Chadaille Bitshiabu22.0 14.66  
WBR       Festy Ebosele       24.0 14.31  
DM        Samson Baidoo       23.0 15.34  
DM        Luca Lipani         22.0 15.0   
WBL       Davide Bartesaghi   21.0 14.75  
AMC       Ruben van Bommel    22.0 14.5   
AMC       Assan Ouédraogo     21.0 14.03  
ST        Jovan Zivkovic      21.0 14.25  

Total Rating for Team 1: 158.62

Best Team 2:
Position  Name                Age  Rating 
GK        Ricard Poch         18.0 11.53  
DC        Travis Akomeah      21.0 14.28  
DC        Ilario Monterisi    25.0 14.16  
DC        Owen Otasowie       26.0 14.06  
WBR       Emil Holm           27.0 13.97  
DM        Gedson Fernandes    28.0 14.62  
DM        Sotiris Alexandropoulos25.0 14.59  
WBL       El Hadji Malick Diouf22.0 14.25  
AMC       Arnaut Danjuma      